# Generate User Profile Summaries with Mistral-7B
**AdRec-GenAI / IEEE 2026 Paper**

This notebook generates natural-language summaries of each user's watch history using Mistral-7B-Instruct-v0.3, then saves the result as `user_generative_summaries.json` to your Google Drive.

**Before running:**
1. Set Runtime → Change runtime type → **A100 GPU** (or T4)
2. Upload to Google Drive (`My Drive/AdRec-GenAI/data/`):
   - `big_matrix.csv` (1.0 GB)
   - `kuairec_caption_category.csv` (25 MB)
3. Run all cells top to bottom

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf
print('Done')

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR  = '/content/drive/MyDrive/AdRec-GenAI/data'
OUT_DIR   = '/content/drive/MyDrive/AdRec-GenAI/embeddings'
os.makedirs(OUT_DIR, exist_ok=True)

# Verify files exist
for f in ['big_matrix.csv', 'kuairec_caption_category.csv']:
    path = os.path.join(DATA_DIR, f)
    exists = os.path.exists(path)
    size = os.path.getsize(path) // (1024*1024) if exists else 0
    print(f'  {f}: {"✅ " + str(size) + " MB" if exists else "❌ NOT FOUND — upload to Drive first"}')

In [ ]:
# ── Cell 3: Load Mistral-7B with 4-bit quantization ───────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Loading {MODEL_ID} in 4-bit...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model.eval()
print('Model loaded ✅')

In [ ]:
# ── Cell 4: Build raw user profiles (same logic as local pipeline) ─────────────
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

TOP_K         = 20
ITEM_SEP      = '; '
DEDUPLICATE   = True
FALLBACK      = 'no description'
FIELD_SEP     = ' | '
FIELDS = [
    ('manual_cover_text', 'cover'),
    ('caption',           'caption'),
    ('topic_tag',         'topics'),
    ('first_level_category_name',  'category1'),
    ('second_level_category_name', 'category2'),
]

def build_text(row):
    parts = []
    for col, prefix in FIELDS:
        if col in row and isinstance(row[col], str) and row[col].strip():
            parts.append(f'{prefix}: {row[col].strip()}')
    return FIELD_SEP.join(parts) if parts else FALLBACK

print('Loading item metadata...')
meta_df = pd.read_csv(os.path.join(DATA_DIR, 'kuairec_caption_category.csv'),
                      sep=None, engine='python', on_bad_lines='skip')
meta_df.columns = [c.strip() for c in meta_df.columns]
vid_col = next(c for c in ['video_id','item_id','id'] if c in meta_df.columns)
meta_df[vid_col] = pd.to_numeric(meta_df[vid_col], errors='coerce')
meta_df = meta_df.dropna(subset=[vid_col])
meta_df[vid_col] = meta_df[vid_col].astype(int)
id2text = {int(row[vid_col]): build_text(row) for _, row in meta_df.iterrows()}
print(f'  {len(id2text):,} items loaded')

print('Loading big_matrix...')
big_matrix = pd.read_csv(
    os.path.join(DATA_DIR, 'big_matrix.csv'),
    dtype={'user_id': 'int32', 'video_id': 'int32'},
    usecols=['user_id', 'video_id', 'watch_ratio'],
)
print(f'  {big_matrix["user_id"].nunique():,} users, {len(big_matrix):,} interactions')

print('Building raw profiles...')
sorted_df = big_matrix.sort_values('watch_ratio', ascending=False)
groups = sorted_df.groupby('user_id')
user_ids, raw_texts = [], []
for uid, grp in tqdm(groups, desc='Profiles'):
    top_items = grp['video_id'].head(TOP_K).tolist()
    descs = [id2text.get(int(v), FALLBACK) for v in top_items]
    if DEDUPLICATE:
        seen, unique = set(), []
        for d in descs:
            if d not in seen: seen.add(d); unique.append(d)
        descs = unique
    user_ids.append(int(uid))
    raw_texts.append(ITEM_SEP.join(descs))
print(f'Built {len(user_ids):,} profiles ✅')

In [ ]:
# ── Cell 5: Generate summaries ─────────────────────────────────────────────────
import json, time

CACHE_PATH     = os.path.join(OUT_DIR, 'user_generative_summaries.json')
MAX_NEW_TOKENS = 80
TEMPERATURE    = 0.2
SAVE_EVERY     = 50    # save to Drive every N users

PROMPT_TEMPLATE = (
    'You are analyzing a short-video platform user. '
    'Based on the following videos they watched most, write a 2-sentence '
    'summary of their interests in plain English. Be concise.\n'
    'Videos: {raw_profile}\nSummary:'
)

# Load existing cache (resumable)
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        cache = json.load(f)
    print(f'Resuming: {len(cache["summaries"]):,} users already done')
else:
    cache = {'model': 'mistralai/Mistral-7B-Instruct-v0.3',
             'backend': 'colab_transformers',
             'generated_at': pd.Timestamp.utcnow().isoformat(),
             'summaries': {}}

summaries = cache['summaries']
remaining = [(uid, txt) for uid, txt in zip(user_ids, raw_texts)
             if str(uid) not in summaries]
print(f'{len(remaining):,} users to process')

def generate_summary(raw_profile: str) -> str:
    prompt = PROMPT_TEMPLATE.format(raw_profile=raw_profile)
    # Use chat template for Mistral instruct
    messages = [{'role': 'user', 'content': prompt}]
    encoded = tokenizer.apply_chat_template(
        messages, return_tensors='pt', add_generation_prompt=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0][encoded.shape[1]:], skip_special_tokens=True)
    return decoded.strip()

errors = 0
for i, (uid, raw_text) in enumerate(tqdm(remaining, desc='Generating')):
    try:
        summary = generate_summary(raw_text)
    except Exception as e:
        errors += 1
        summary = raw_text[:200]
        print(f'  Error user {uid}: {e}')

    summaries[str(uid)] = summary

    # Save to Drive every SAVE_EVERY users
    if (i + 1) % SAVE_EVERY == 0:
        with open(CACHE_PATH, 'w') as f:
            json.dump(cache, f, indent=2, ensure_ascii=False)

# Final save
with open(CACHE_PATH, 'w') as f:
    json.dump(cache, f, indent=2, ensure_ascii=False)

print(f'\n✅ Done! {len(summaries):,} summaries saved')
print(f'Errors: {errors}')
print(f'File: {CACHE_PATH}')

In [ ]:
# ── Cell 6: Verify and show examples ──────────────────────────────────────────
print(f'Total summaries: {len(summaries):,}')
print(f'File size: {os.path.getsize(CACHE_PATH) / 1024:.1f} KB')
print()
print('Example summaries:')
for uid_str, summary in list(summaries.items())[:5]:
    print(f'  User {uid_str}: {summary}')
    print()

## Done! Next steps on your local machine

1. **Download** `user_generative_summaries.json` from Google Drive
   - It's in `My Drive/AdRec-GenAI/embeddings/`

2. **Put it in your project:**
   ```
   AdRec-GenAI/kuairec/embeddings/user_generative_summaries.json
   ```

3. **Encode into embeddings locally:**
   ```bash
   cd /Users/tanushreenepal/Desktop/AdRec-GenAI
   python LLM-rec/src/build_user_llm_embeddings.py --use_generative
   ```

4. **Train and compare models:**
   ```bash
   python LLM-rec/src/run_all.py --user_emb generative
   ```